In [1]:
import sys
import os
import numpy as np
sys.path.append(os.path.abspath(".."))

from src.trigram import TrigramModel
from src.normalisation import normalise_text

## Trigram model's probabilities with the two-character history t h

In [2]:
# Load the data
with open("../data/train.en.txt") as f:
    text = f.read()

# Normalise the data
sentences = normalise_text(text)

In [3]:
# Build and fit the model
trigram_model = TrigramModel()
trigram_model.fit(sentences)

th_probs = {k: v / trigram_model.history_counts["th"] for k, v in trigram_model.counts["th"].items()}
dict(sorted(th_probs.items(), key=lambda item: item[1], reverse=True)) # print sorted dict

{'e': 0.6807593914097213,
 ' ': 0.10158879763026794,
 'a': 0.08307526592163728,
 'i': 0.04470176383465733,
 'o': 0.037969570486064357,
 'r': 0.033862932543422646,
 'n': 0.004577891477043221,
 's': 0.0032314528073246265,
 '$': 0.002894843139894978,
 'u': 0.00235626767200754,
 'y': 0.0019523360710919618,
 'w': 0.0006732193348592972,
 'd': 0.0006058974013733675,
 'l': 0.0005385754678874377,
 'm': 0.0004039316009155783,
 'c': 0.00026928773394371884,
 't': 0.00020196580045778916,
 'p': 0.00013464386697185942,
 '–': 6.732193348592971e-05,
 'b': 6.732193348592971e-05,
 'f': 6.732193348592971e-05}

We observe that the letter "e" follows with the highest probability, and then the space character and then a, which is around what we would expect intuitively

## Generated  Trigram language

In [4]:
gen_text = trigram_model.generate(seed_text=4)

print(gen_text)

the a inoter thimparinchatienturval dia a the comers ok stromers


## Perplexity Analysis

### 1) For English training text

In [5]:
perplexity = trigram_model.perplexity(sentences)
print(f"Model Perplexity: {perplexity}")

Model Perplexity: 7.302904870502697


### 2) Train 5 different Language Models and evaluate Perplexity of each

In [6]:
# English Model
with open("../data/train.en.txt") as f:
    eng_text = f.read()

eng_sentences = normalise_text(eng_text)
eng_model = TrigramModel()
eng_model.fit(eng_sentences)

# Afrikaans Model
with open("../data/train.af.txt") as f:
    af_text = f.read()

af_sentences = normalise_text(af_text)
af_model = TrigramModel()
af_model.fit(af_sentences)

# Dutch Model
with open("../data/train.nl.txt") as f:
    nl_text = f.read()

nl_sentences = normalise_text(nl_text)
nl_model = TrigramModel()
nl_model.fit(nl_sentences)

# Xhosa model
with open("../data/train.xh.txt") as f:
    xh_text = f.read()

xh_sentences = normalise_text(xh_text)
xh_model = TrigramModel()
xh_model.fit(xh_sentences)

# Zulu Model
with open("../data/train.zu.txt") as f:
    zu_text = f.read()

zu_sentences = normalise_text(zu_text)
zu_model = TrigramModel()
zu_model.fit(zu_sentences)


In [7]:
# Determine perplexity on validation sets

# import validation data
with open("../data/val.en.txt") as f:
    eng_val_text = f.read()
eng_val_sentences = normalise_text(eng_val_text)

with open("../data/val.af.txt") as f:
    af_val_text = f.read()
af_val_sentences = normalise_text(af_val_text)

with open("../data/val.nl.txt") as f:
    nl_val_text = f.read()
nl_val_sentences = normalise_text(nl_val_text)

with open("../data/val.xh.txt") as f:
    xh_val_text = f.read()
xh_val_sentences = normalise_text(xh_val_text)

with open("../data/val.zu.txt") as f:
    zu_val_text = f.read()
zu_val_sentences = normalise_text(zu_val_text)

def compute_perplexities(model):
    eng_perp = np.round(model.perplexity(eng_val_sentences), 2)
    af_perp = np.round(model.perplexity(af_val_sentences), 2)
    nl_perp = np.round(model.perplexity(nl_val_sentences), 2)
    xh_perp = np.round(model.perplexity(xh_val_sentences), 2)
    zu_perp = np.round(model.perplexity(zu_val_sentences), 2)

    return {"eng": eng_perp, "af": af_perp, "nl": nl_perp, "xh": xh_perp, "zu": zu_perp}

eng_perps = compute_perplexities(eng_model)
af_perps = compute_perplexities(af_model)
nl_perps = compute_perplexities(nl_model)
xh_perps = compute_perplexities(xh_model)
zu_perps = compute_perplexities(zu_model)


In [8]:
print("English Model Perplexities:")
print({k: float(v) for k, v in eng_perps.items()})

print("Afrikaans Model Perplexities:")
print({k: float(v) for k, v in af_perps.items()})

print("Dutch Model Perplexities:")
print({k: float(v) for k, v in nl_perps.items()})

print("Xhosa Model Perplexities:")
print({k: float(v) for k, v in xh_perps.items()})

print("Zulu Model Perplexities:")
print({k: float(v) for k, v in zu_perps.items()})

English Model Perplexities:
{'eng': 8.62, 'af': 63.14, 'nl': 102.45, 'xh': 741.77, 'zu': 823.71}
Afrikaans Model Perplexities:
{'eng': 32.03, 'af': 9.6, 'nl': 22.96, 'xh': 349.01, 'zu': 369.82}
Dutch Model Perplexities:
{'eng': 21.13, 'af': 14.4, 'nl': 9.11, 'xh': 637.82, 'zu': 595.74}
Xhosa Model Perplexities:
{'eng': 26.89, 'af': 106.71, 'nl': 161.09, 'xh': 9.82, 'zu': 11.7}
Zulu Model Perplexities:
{'eng': 51.98, 'af': 204.26, 'nl': 266.16, 'xh': 15.32, 'zu': 11.84}
